# Build directed and mutual graphs

Load every generated nomination network once, preserve the directed choices,
and derive the reciprocal friendship projection.


In [ ]:
import pickle
import re
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "code":
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "outputs" / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
ANES_CSV = BASE_DIR / "data" / "anes_100.csv"
RAW_NETWORK_DIR = BASE_DIR / "data" / "sample_100_v3"



In [ ]:
CSV_FILE = ANES_CSV
NETWORK_FOLDER = RAW_NETWORK_DIR
DIRECTED_PKL = DATA_DIR / "graphs_sample_100_v3_directed.pkl"
MUTUAL_PKL = DATA_DIR / "graphs_sample_100_v3_mutual.pkl"
DIAGNOSTICS_CSV = DATA_DIR / "graph_build_diagnostics_sample_100_v3.csv"

VOTE_ATTR = "w2presvtwho"
REQUIRED_COLS = [
    "caseid",
    "profile_gender",
    "profile_age",
    "profile_age_bucket",
    "profile_racethnicity",
    "profile_educ5",
    "profile_state",
    "profile_region4",
    "profile_relig",
    "w2presvtwho",
    "weight_post",
]


In [ ]:
def load_personas(csv_file, required_cols):
    """Load and validate the authoritative 100-person roster."""
    df = pd.read_csv(csv_file)

    missing_columns = sorted(set(required_cols) - set(df.columns))
    if missing_columns:
        raise ValueError(f"Missing persona columns: {missing_columns}")

    df = df[required_cols].copy()
    if df["caseid"].isna().any():
        raise ValueError("Persona roster contains missing case IDs.")
    df["caseid"] = df["caseid"].astype(int)
    if df["caseid"].duplicated().any():
        duplicates = df.loc[df["caseid"].duplicated(), "caseid"].tolist()
        raise ValueError(f"Duplicate persona IDs: {duplicates}")
    if df[required_cols].isna().any().any():
        missing = df[required_cols].isna().sum()
        raise ValueError(f"Missing persona attributes:\n{missing[missing.gt(0)]}")
    if len(df) != 100:
        raise ValueError(f"Expected 100 personas, found {len(df)}.")

    return df.to_dict(orient="records")


personas = load_personas(CSV_FILE, REQUIRED_COLS)
persona_dict = {persona["caseid"]: persona for persona in personas}
persona_ids = set(persona_dict)

print(f"Validated personas: {len(personas)}")
print("\nVote attribute distribution:")
print(pd.Series([persona[VOTE_ATTR] for persona in personas]).value_counts().to_string())


In [ ]:
def load_directed_graph(edge_file, persona_dict):
    """Load one complete directed nomination network."""
    G = nx.DiGraph()
    G.add_nodes_from((pid, attrs) for pid, attrs in persona_dict.items())

    seen_sources = set()
    with Path(edge_file).open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            match = re.fullmatch(r"\s*(\d+)\s*:\s*(.*?)\s*", line)
            if not match:
                raise ValueError(f"{edge_file}:{line_number}: malformed row")

            source = int(match.group(1))
            if source not in persona_dict:
                raise ValueError(f"{edge_file}:{line_number}: unknown source {source}")
            if source in seen_sources:
                raise ValueError(f"{edge_file}:{line_number}: duplicate source {source}")
            seen_sources.add(source)

            target_text = match.group(2).strip()
            target_tokens = [] if not target_text else [token.strip() for token in target_text.split(",")]
            if any(not token.isdigit() for token in target_tokens):
                raise ValueError(f"{edge_file}:{line_number}: malformed target list")

            targets = [int(token) for token in target_tokens]
            if len(targets) != len(set(targets)):
                raise ValueError(f"{edge_file}:{line_number}: duplicate target")
            if source in targets:
                raise ValueError(f"{edge_file}:{line_number}: self-nomination {source}")

            unknown_targets = sorted(set(targets) - set(persona_dict))
            if unknown_targets:
                raise ValueError(
                    f"{edge_file}:{line_number}: unknown targets {unknown_targets}"
                )
            G.add_edges_from((source, target) for target in targets)

    missing_sources = sorted(set(persona_dict) - seen_sources)
    if missing_sources:
        raise ValueError(f"{edge_file}: missing source rows {missing_sources}")
    if len(seen_sources) != len(persona_dict):
        raise ValueError(f"{edge_file}: unexpected number of source rows")

    return G


In [ ]:
directed_graphs = {}
mutual_graphs = {}
diag_rows = []

model_folders = sorted(
    path for path in NETWORK_FOLDER.iterdir()
    if path.is_dir() and not path.name.startswith(".")
)

for model_path in model_folders:
    model = model_path.name
    directed_graphs[model] = {}
    mutual_graphs[model] = {}

    for edge_file in sorted(model_path.glob("*.txt")):
        fname = edge_file.name
        directed = load_directed_graph(edge_file, persona_dict)
        mutual = directed.to_undirected(reciprocal=True)

        reciprocal_pairs = {
            frozenset((source, target))
            for source, target in directed.edges()
            if directed.has_edge(target, source)
        }
        if set(map(frozenset, mutual.edges())) != reciprocal_pairs:
            raise ValueError(f"{edge_file}: invalid mutual projection")

        directed_graphs[model][fname] = directed
        mutual_graphs[model][fname] = mutual

        n = mutual.number_of_nodes()
        directed_out_isolates = sum(directed.out_degree(node) == 0 for node in directed)
        mutual_isolates = nx.number_of_isolates(mutual)
        largest_component = max(nx.connected_components(mutual), key=len)

        diag_rows.append({
            "model": model,
            "file": fname,
            "nodes": n,
            "directed_arcs": directed.number_of_edges(),
            "mutual_edges": mutual.number_of_edges(),
            "directed_out_isolates": directed_out_isolates,
            "mutual_isolates": mutual_isolates,
            "mutual_isolate_%": round(mutual_isolates / n * 100, 2),
            "directed_density": round(nx.density(directed), 6),
            "mutual_density": round(nx.density(mutual), 6),
            "reciprocity": round(nx.reciprocity(directed), 4),
            "mutual_lcc_ratio": round(len(largest_component) / n, 4),
        })

diag_df = pd.DataFrame(diag_rows)
print(
    f"Built {len(diag_df):,} directed/mutual graph pairs from "
    f"{len(directed_graphs)} models."
)
display(diag_df.head())


In [ ]:
summary = (
    diag_df
    .groupby("model")
    .agg(
        runs=("file", "count"),
        nodes=("nodes", "first"),
        directed_arcs_mean=("directed_arcs", "mean"),
        mutual_edges_mean=("mutual_edges", "mean"),
        directed_density_mean=("directed_density", "mean"),
        mutual_density_mean=("mutual_density", "mean"),
        directed_out_isolates_mean=("directed_out_isolates", "mean"),
        mutual_isolates_mean=("mutual_isolates", "mean"),
        reciprocity_mean=("reciprocity", "mean"),
        mutual_lcc_ratio_mean=("mutual_lcc_ratio", "mean"),
    )
    .round(4)
)

print(summary.to_string())

if len(diag_df) != 1000:
    raise ValueError(f"Expected 1,000 networks, found {len(diag_df)}.")
if not summary["runs"].eq(200).all():
    raise ValueError(f"Expected 200 networks per model:\n{summary['runs']}")
if not diag_df["nodes"].eq(len(persona_dict)).all():
    raise ValueError("At least one graph has an inconsistent node roster.")
if diag_df[["model", "file"]].duplicated().any():
    raise ValueError("Duplicate model/file identifiers detected.")

reciprocity_error = (
    2 * diag_df["mutual_edges"] / diag_df["directed_arcs"]
    - diag_df["reciprocity"]
).abs().max()
if reciprocity_error > 5.1e-5:
    raise ValueError("Reciprocity and mutual-edge counts are inconsistent.")

print("Validation passed: 1,000 complete graph pairs on the same 100-person roster.")


In [ ]:
for output_path, graph_collection in [
    (DIRECTED_PKL, directed_graphs),
    (MUTUAL_PKL, mutual_graphs),
]:
    with output_path.open('wb') as f:
        pickle.dump(graph_collection, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved: {output_path} ({output_path.stat().st_size / 1_048_576:.2f} MB)")

diag_df.to_csv(DIAGNOSTICS_CSV, index=False)
print(f"Diagnostics: {DIAGNOSTICS_CSV}")
print(f"Models: {list(mutual_graphs)}")
print(f"Files: { {model: len(runs) for model, runs in mutual_graphs.items()} }")
